# Lab 3 - N-Grams

Task: generate tweets with an n-gram model. I build a bigram model (MLE) with NLTK on tweets from Pakistan, then generate tweets, evaluate it and check a few probabilities.

Libraries used: nltk, pandas, kagglehub, emoji. Install them first with `pip install nltk pandas kagglehub emoji ipywidgets` if they are missing.

In [1]:
import warnings
warnings.filterwarnings("ignore")  # hide the tqdm progress bar warning

import os
import random
import re
from itertools import chain

import emoji
import kagglehub
import pandas as pd
from nltk.lm import MLE
from nltk.lm.preprocessing import pad_both_ends, padded_everygram_pipeline
from nltk.util import ngrams

## 1. Load the data

In [2]:
path = kagglehub.dataset_download("adizafar/large-random-tweets-from-pakistan")
csv_file = [os.path.join(r, f) for r, d, fs in os.walk(path) for f in fs if f.endswith(".csv")][0]
print(csv_file)

C:\Users\GAMING\.cache\kagglehub\datasets\adizafar\large-random-tweets-from-pakistan\versions\2\Random Tweets from Pakistan- Cleaned- Anonymous.csv


In [3]:
# the file has a few bad bytes, so utf-8 needs errors="replace"
df = pd.read_csv(csv_file, encoding="utf-8", encoding_errors="replace")
print(df.shape)
print(df.columns.tolist())
df[["full_text"]].head()

(202202, 7)
['Unnamed: 0', 'created_at_tweet', 'full_text', 'retweet_count', 'favorite_count', 'reply_count', 'location']


,full_text
0,تیرا لیڈر میرا لیڈر نواز شریف نواز شریف❤️ کیا...
1,"Happy birthday to my brother n boss , May you ..."
2,❤️❤️
3,`suspicious °jikook au jimin'in yaşadığı kasab...
4,Speaking of @SpiderMan... 😂 https://t.co/uGJ...


## 2. Pre-processing

For every tweet I do the following:
- lowercase
- remove `RT`
- remove websites
- remove mentions (`@name`)
- remove hashtags
- remove emojis

After that I keep only English letters (and apostrophes), so Urdu and other languages are dropped, and I throw away tweets shorter than 4 words.

In [4]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"\brt\b", " ", text)               # remove RT
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # remove websites
    text = re.sub(r"@\w+", " ", text)                  # remove mentions
    text = re.sub(r"#\w+", " ", text)                  # remove hashtags
    text = emoji.replace_emoji(text, replace=" ")      # remove emojis
    text = re.sub(r"[^a-z' ]", " ", text)              # keep only english letters
    text = re.sub(r"\s+", " ", text).strip()
    return text

print(clean_tweet("RT @user: Pakistan is great!! \U0001F602 #cricket https://t.co/abc"))

pakistan is great


In [5]:
df["clean"] = df["full_text"].apply(clean_tweet)
df = df[df["clean"].str.split().str.len() >= 4]
df = df.drop_duplicates(subset="clean")
print("tweets left:", len(df))
df[["full_text", "clean"]].head()

tweets left: 80929


,full_text,clean
1,"Happy birthday to my brother n boss , May you ...",happy birthday to my brother n boss may you ha...
3,`suspicious °jikook au jimin'in yaşadığı kasab...,suspicious jikook au jimin'in ya ad kasabada a...
6,Jb tm moon lover ho gy to sooraj to tappy ga na 😒,jb tm moon lover ho gy to sooraj to tappy ga na
7,Alexa skip to 2022 now,alexa skip to now
8,"But this was one of my first shoots, about 6 y...",but this was one of my first shoots about year...


## 3. Build the MLE bigram model

I split the tweets 80% train and 20% test, add the start and end padding, and train a bigram model with `MLE`.

In [6]:
tweets = df["clean"].tolist()
random.seed(42)
random.shuffle(tweets)

split = int(len(tweets) * 0.8)
train_tweets, test_tweets = tweets[:split], tweets[split:]
print("train:", len(train_tweets), "test:", len(test_tweets))

train: 64743 test: 16186


In [7]:
train_tokens = [t.split() for t in train_tweets]

train_ngrams, vocab = padded_everygram_pipeline(2, train_tokens)

lm = MLE(2)
lm.fit(train_ngrams, vocab)
print("vocabulary size:", len(lm.vocab))

vocabulary size: 66775


## 4. Generate tweets

Each tweet starts from a seed word and the model picks the next word using the bigram probabilities. It stops at the end symbol or after 20 words.

In [8]:
def generate_tweet(seed, max_words=20, random_seed=1):
    words = []
    for w in lm.generate(max_words, text_seed=[seed], random_seed=random_seed):
        if w == "</s>":
            break
        if w != "<s>":
            words.append(w)
    return " ".join([seed] + words)

for i, seed in enumerate(["pakistan", "i", "happy", "the", "we"]):
    print(i + 1, "-", generate_tweet(seed, random_seed=i))

1 - pakistan team rasgulla am facing loadshedding in the fourth of peace start making for the parliament could the world should take
2 - i believe them the exams included in reforming the bri
3 - happy watch when are at the reports new episodes cid new opportunities and it's heavy price you why its introducing him
4 - the end of journalism in promoting and a strong champ
5 - we aren't blind eye care


## 5. Evaluate the model

I use perplexity (lower is better). First on bigrams from the test tweets, then on bigrams from the training tweets.

In [9]:
def to_bigrams(token_lists):
    return list(chain.from_iterable(ngrams(pad_both_ends(t, n=2), 2) for t in token_lists))

test_bigrams = to_bigrams([t.split() for t in test_tweets[:2000]])
train_bigrams = to_bigrams(train_tokens[:2000])

unseen = sum(1 for a, b in test_bigrams if lm.score(b, [a]) == 0)
print("test bigrams:", len(test_bigrams))
print("unseen test bigrams: %.1f%%" % (100 * unseen / len(test_bigrams)))

print("perplexity on test bigrams:", lm.perplexity(test_bigrams))
print("perplexity on train bigrams:", lm.perplexity(train_bigrams))

test bigrams: 38420
unseen test bigrams: 35.3%
perplexity on test bigrams: inf
perplexity on train bigrams: 79.47708519951505


The test perplexity is `inf`. MLE gives probability 0 to any bigram it did not see in training, and a single zero makes the whole perplexity infinite. The training bigrams have a finite value because the model saw them. A smoothed model (like Laplace) would fix this.

## 6. Probability of the bigram (pakistan is)

$P(\text{is} \mid \text{pakistan}) = \frac{count(pakistan\ is)}{count(pakistan)}$

In [10]:
print("count(pakistan is):", lm.counts[["pakistan"]]["is"])
print("count(pakistan):", lm.counts["pakistan"])
print("P(is | pakistan) =", lm.score("is", ["pakistan"]))

count(pakistan is): 285
count(pakistan): 6503
P(is | pakistan) = 0.04382592649546363


## 7. Perplexity of the word (pakistan)

Here I use the single word `pakistan` as the test n-gram. Its probability is the unigram probability, and the perplexity is 1 divided by that probability.

In [11]:
p_word = lm.score("pakistan")
print("P(pakistan) =", p_word)
print("perplexity of 'pakistan':", lm.perplexity([("pakistan",)]))
print("1 / P(pakistan)         :", 1 / p_word)

P(pakistan) = 0.005032693547425959
perplexity of 'pakistan': 198.70075349838544
1 / P(pakistan)         : 198.70075349838535
